# 03 — Ekstraksi deep embeddings dan baseline

GPU dianjurkan untuk ekstraksi. Memuat checkpoint per fold dan mengekstraksi embedding berukuran 768 tanpa augmentasi. Baseline LR/LGBM memakai hanya train untuk fit.

**Pipeline revisi untuk artikel ilmiah.** Hasil tidak otomatis identik dengan skripsi 2026 karena split, kontrol, dan beberapa transformasi diperbarui. Jalankan notebook menurut urutan nomor.

In [ ]:
%pip -q install "timm>=1.0,<2" "lightgbm>=4,<5" scikit-learn joblib

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, os

drive.mount('/content/drive')
PROJECT=Path(os.environ.get('FFE_PROJECT','/content/drive/MyDrive/tugas_akhir/FFE_Publication_2026'))
assert PROJECT.exists(), f'Folder proyek tidak ditemukan: {PROJECT}'
os.environ['FFE_ROOT']=str(PROJECT)
sys.path.insert(0,str(PROJECT/'02_notebooks'))
from ffe_core import ROOT, paths, seed_all, ensure_dataset_local, load_manifest, load_split
P=paths(); seed_all()
print('Project:',ROOT)

In [ ]:
import torch, numpy as np,pandas as pd,joblib
from torch.utils.data import DataLoader
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from ffe_core import ImageDataset,get_transform,extract_feature_checkpoint,save_pack,read_pack,save_prediction,metric_row
from tqdm.auto import tqdm
ensure_dataset_local()
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE=32

## 1. Ekstraksi embedding dengan backbone spesifik fold

In [ ]:
for fold in range(1,6):
 print('Fold',fold)
 ckpt=P['checkpoints']/f'swin_fold_{fold:02d}_best.pt'
 if not ckpt.exists():raise FileNotFoundError('Run notebook 02 first: '+str(ckpt))
 model=extract_feature_checkpoint(ckpt,DEVICE)
 out=P['features']/f'fold_{fold:02d}';out.mkdir(exist_ok=True)
 for part in ('train','val','test'):
  d=load_split(fold,part)
  loader=DataLoader(ImageDataset(d,get_transform(False)),batch_size=BATCH_SIZE,shuffle=False,num_workers=2)
  xs=[]
  with torch.no_grad():
   for x,_ in tqdm(loader,desc=f'{part} fold {fold}',leave=False):
    with torch.autocast(device_type='cuda',enabled=(DEVICE.type=='cuda')):emb=model(x.to(DEVICE))
    xs.append(emb.float().cpu().numpy())
  X=np.concatenate(xs)
  if X.shape!=(len(d),768):raise RuntimeError(f'Unexpected embedding dimension: {X.shape}')
  save_pack(out/f'{part}_deep.npz',X,d)
  read_pack(out/f'{part}_deep.npz',d)
 del model
 if DEVICE.type=='cuda':torch.cuda.empty_cache()

## 2. Baseline: Linear classifier dan LightGBM di atas embedding
Kedua baseline menggunakan pembagian data identik dengan eksperimen fusi.

In [ ]:
for fold in range(1,6):
 d={part:load_split(fold,part) for part in ('train','val','test')}
 pack={part:read_pack(P['features']/f'fold_{fold:02d}'/f'{part}_deep.npz',d[part]) for part in d}
 X,y=pack['train']['X'],pack['train']['y']
 models={
  'deep_lr':make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000,class_weight='balanced',random_state=42)),
  'deep_lgbm':LGBMClassifier(n_estimators=300,learning_rate=.05,num_leaves=31,max_depth=-1,
                             class_weight='balanced',random_state=42,verbosity=-1,n_jobs=-1)
 }
 for name,m in models.items():
  m.fit(X,y)
  joblib.dump(m,P['models']/f'{name}_fold_{fold:02d}.joblib')
  proba=m.predict_proba(pack['test']['X'])
  pred=proba.argmax(1)
  save_prediction(P['metrics']/f'pred_{name}_fold_{fold:02d}.csv',d['test'],pred,proba,name,fold)
  print(fold,name,metric_row(pack['test']['y'],pred))
print('Notebook 03 selesai.')